<a href="https://colab.research.google.com/github/NaydelinAidee/Procesos-Estocasticos/blob/main/DanoneExamen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd

# Cargar datos
file_path = 'Examen practico resuelto p(1).xlsx'
xls = pd.ExcelFile(file_path)
df_venta = pd.read_excel(xls, sheet_name='Venta')

#columna 'Concatenado'
df_venta['Concatenado'] = (
    df_venta['UDN\nCuota factor con rango v4'].astype(str) +
    df_venta['Canal de venta\nCuota factor con rango v4'].astype(str) +
    df_venta['Tipo de Empleado\nCatálogo de Empleados'].astype(str) +
    df_venta['Vendedor solo\nLiquidaciones'].astype(str) +
    df_venta['Tipo día\nLiquidaciones'].astype(str)
)

#Calcular Alcance Diario
# Unidades por registro / Cuota total del empleado
df_venta['Alcance Diario'] = (
    df_venta['Unidades Liquidaciones'] /
    df_venta['Cuota\nCuota factor con rango v4']
)

#Organizar columnas para mostrar
resultado_diario = df_venta[[
    'Concatenado',
    'Empleado\nLiquidaciones',
    'Unidades Liquidaciones',
    'Cuota\nCuota factor con rango v4',
    'Alcance Diario'
]]

# Mostrar resultado formateado
print(resultado_diario.head(45).to_string(
    formatters={'Alcance Diario': '{:.2%}'.format},
    index=False
))

Concatenado  Empleado\nLiquidaciones  Unidades Liquidaciones  Cuota\nCuota factor con rango v4 Alcance Diario
 BASL112501                   370115                   199.0                          192.3980        103.43%
 BASL112501                   370115                     1.2                          192.3980          0.62%
 BASL146501                   370169                   199.0                          192.3980        103.43%
 BASL146501                   370169                     1.2                          192.3980          0.62%
 BASL146501                   371936                    68.0                          154.3779         44.05%
 BASL112501                 10514641                    68.0                          154.3779         44.05%
 BASL146501                   371936                   116.0                          154.3779         75.14%
 BASL112501                 10514641                   116.0                          154.3779         75.14%
 BASL11250

In [22]:
# Función de Tarifa Base
def obtener_tarifa_base(row, df_factores):
    concatenado = row['Concatenado']
    alcance = row['Alcance Diario']

    # Buscamos la fila en Factores
    factor_row = df_factores[df_factores['CONCATENADO'] == concatenado]

    if factor_row.empty:
        return 0

    # Lógica de rangos
    if alcance < 0.80: col = '0 -79.9%'
    elif alcance < 0.86: col = '80 - 85.9%'
    elif alcance < 0.91: col = '86 - 90.9%'
    elif alcance < 0.98: col = '91 - 97.9%'
    elif alcance <= 1.02: col = '98 - 101.9%'
    else: col = '>102%'

    return factor_row[col].values[0]

# Aplicar cálculo
df_venta['Tarifa Base'] = df_venta.apply(lambda row: obtener_tarifa_base(row, df_factores), axis=1)

#Visualizar
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df_venta[['Concatenado', 'Alcance Diario', 'Tarifa Base']].head(45))

   Concatenado  Alcance Diario  Tarifa Base
0   BASL112501        1.034314         2.52
1   BASL112501        0.006237         0.89
2   BASL146501        1.034314         3.40
3   BASL146501        0.006237         1.24
4   BASL146501        0.440478         1.24
5   BASL112501        0.440478         0.89
6   BASL146501        0.751403         1.24
7   BASL112501        0.751403         0.89
8   BASL112501        1.147149         2.52
9   BASL146501        1.147149         3.40
10  BASL146501        0.616133         1.24
11  BASL112501        0.616133         0.89
12  BASL146501        0.478550         1.24
13  BASL112501        0.478550         0.89
14  BASL112501        1.182105         2.52
15  BASL112501        0.008866         0.89
16  BASL146501        1.182105         3.40
17  BASL146501        0.008866         1.24
18  BASL146501        0.313062         1.24
19  BASL112501        0.313062         0.89
20  BASL146501        0.712063         1.24
21  BASL112501        0.712063  

In [24]:
# Calcular Pago Base
df_venta['Pago Base'] = df_venta['Unidades Liquidaciones'] * df_venta['Tarifa Base']

# Mostrar resultados
print(df_venta[['Concatenado', 'Unidades Liquidaciones', 'Tarifa Base', 'Pago Base']].head(45).to_string())

   Concatenado  Unidades Liquidaciones  Tarifa Base  Pago Base
0   BASL112501                   199.0         2.52    501.480
1   BASL112501                     1.2         0.89      1.068
2   BASL146501                   199.0         3.40    676.600
3   BASL146501                     1.2         1.24      1.488
4   BASL146501                    68.0         1.24     84.320
5   BASL112501                    68.0         0.89     60.520
6   BASL146501                   116.0         1.24    143.840
7   BASL112501                   116.0         0.89    103.240
8   BASL112501                   239.0         2.52    602.280
9   BASL146501                   239.0         3.40    812.600
10  BASL146501                   103.0         1.24    127.720
11  BASL112501                   103.0         0.89     91.670
12  BASL146501                    80.0         1.24     99.200
13  BASL112501                    80.0         0.89     71.200
14  BASL112501                   240.0         2.52    

In [25]:
# 'Jugs Excedentes'

df_venta['Jugs Excedentes'] = df_venta.apply(
    lambda row: row['Unidades Liquidaciones'] - (row['Cuota\nCuota factor con rango v4'] * 0.9)
    if row['Alcance Diario'] > 0.9 else 0, axis=1
)

# Opcional: Aseguramos que no haya valores negativos por redondeo
df_venta['Jugs Excedentes'] = df_venta['Jugs Excedentes'].clip(lower=0)

# Imprimimos el resultado para verificar
print(df_venta[['Concatenado', 'Alcance Diario', 'Unidades Liquidaciones', 'Cuota\nCuota factor con rango v4', 'Jugs Excedentes']].head(45).to_string())

   Concatenado  Alcance Diario  Unidades Liquidaciones  Cuota\nCuota factor con rango v4  Jugs Excedentes
0   BASL112501        1.034314                   199.0                          192.3980         25.84180
1   BASL112501        0.006237                     1.2                          192.3980          0.00000
2   BASL146501        1.034314                   199.0                          192.3980         25.84180
3   BASL146501        0.006237                     1.2                          192.3980          0.00000
4   BASL146501        0.440478                    68.0                          154.3779          0.00000
5   BASL112501        0.440478                    68.0                          154.3779          0.00000
6   BASL146501        0.751403                   116.0                          154.3779          0.00000
7   BASL112501        0.751403                   116.0                          154.3779          0.00000
8   BASL112501        1.147149                

In [27]:
# Función para obtener la Tarifa Booster
def obtener_tarifa_booster(row, df_factores):
    concatenado = row['Concatenado']
    alcance = row['Alcance Diario']

    # Buscar la llave en la tabla de factores
    factor_row = df_factores[df_factores['CONCATENADO'] == concatenado]

    # Si no hay coincidencia, devolvemos 0
    if factor_row.empty:
        return 0

    if alcance > 1.01:
        col = 'JUGS ADICIONALES >101%' # Columna 9 en tu matriz de Excel
    else:
        col = 'JUGS ADICIONALES 91-101%' # Columna 8 en tu matriz de Excel

    return factor_row[col].values[0]

# Aplicar la función al DataFrame para crear la columna
df_venta['Tarifa Booster'] = df_venta.apply(lambda row: obtener_tarifa_booster(row, df_factores), axis=1)

# Visualizar el resultado
print(df_venta[['Concatenado', 'Alcance Diario', 'Tarifa Booster']].head(45).to_string())

   Concatenado  Alcance Diario  Tarifa Booster
0   BASL112501        1.034314             4.0
1   BASL112501        0.006237             3.0
2   BASL146501        1.034314             5.0
3   BASL146501        0.006237             3.7
4   BASL146501        0.440478             3.7
5   BASL112501        0.440478             3.0
6   BASL146501        0.751403             3.7
7   BASL112501        0.751403             3.0
8   BASL112501        1.147149             4.0
9   BASL146501        1.147149             5.0
10  BASL146501        0.616133             3.7
11  BASL112501        0.616133             3.0
12  BASL146501        0.478550             3.7
13  BASL112501        0.478550             3.0
14  BASL112501        1.182105             4.0
15  BASL112501        0.008866             3.0
16  BASL146501        1.182105             5.0
17  BASL146501        0.008866             3.7
18  BASL146501        0.313062             3.7
19  BASL112501        0.313062             3.0
20  BASL14650

In [29]:
# Cálculo del Pago Booster= Tarifa Booster * Jugs Excedentes
df_venta['Pago Booster'] = df_venta['Tarifa Booster'] * df_venta['Jugs Excedentes']

# Visualizar el resultado de la operación
print("Cálculo de Pago Booster:")
print(df_venta[['Concatenado', 'Jugs Excedentes', 'Tarifa Booster', 'Pago Booster']].head(45))

Cálculo de Pago Booster:
   Concatenado  Jugs Excedentes  Tarifa Booster  Pago Booster
0   BASL112501         25.84180             4.0    103.367200
1   BASL112501          0.00000             3.0      0.000000
2   BASL146501         25.84180             5.0    129.209000
3   BASL146501          0.00000             3.7      0.000000
4   BASL146501          0.00000             3.7      0.000000
5   BASL112501          0.00000             3.0      0.000000
6   BASL146501          0.00000             3.7      0.000000
7   BASL112501          0.00000             3.0      0.000000
8   BASL112501         51.49166             4.0    205.966640
9   BASL146501         51.49166             5.0    257.458300
10  BASL146501          0.00000             3.7      0.000000
11  BASL112501          0.00000             3.0      0.000000
12  BASL146501          0.00000             3.7      0.000000
13  BASL112501          0.00000             3.0      0.000000
14  BASL112501         57.27507             4

In [31]:
# Cálculo del Total a Pagar=Pago Base + Pago Booster

df_venta['Total a Pagar'] = df_venta['Pago Base'] + df_venta['Pago Booster']

# Visualizar el resumen final con los montos
print("Resumen de Pagos:")
print(df_venta[['Concatenado', 'Pago Base', 'Pago Booster', 'Total a Pagar']].head(45))

Resumen de Pagos:
   Concatenado  Pago Base  Pago Booster  Total a Pagar
0   BASL112501    501.480    103.367200     604.847200
1   BASL112501      1.068      0.000000       1.068000
2   BASL146501    676.600    129.209000     805.809000
3   BASL146501      1.488      0.000000       1.488000
4   BASL146501     84.320      0.000000      84.320000
5   BASL112501     60.520      0.000000      60.520000
6   BASL146501    143.840      0.000000     143.840000
7   BASL112501    103.240      0.000000     103.240000
8   BASL112501    602.280    205.966640     808.246640
9   BASL146501    812.600    257.458300    1070.058300
10  BASL146501    127.720      0.000000     127.720000
11  BASL112501     91.670      0.000000      91.670000
12  BASL146501     99.200      0.000000      99.200000
13  BASL112501     71.200      0.000000      71.200000
14  BASL112501    604.800    229.100280     833.900280
15  BASL112501      1.602      0.000000       1.602000
16  BASL146501    816.000    286.375350    1102

In [36]:
import pandas as pd

#  Agrupar por empleado y sumar los pagos
resumen_empleados = df_venta.groupby('Empleado\nLiquidaciones')[['Pago Base', 'Pago Booster', 'Total a Pagar']].sum().reset_index()

#nombrar las columnas
resumen_empleados.rename(columns={
    'Empleado\nLiquidaciones': 'A quien se le paga',
    'Pago Base': 'Pago Base E',
    'Pago Booster': 'PAGO BOOSTER E',
    'Total a Pagar': 'TOTAL SEMANAL'
}, inplace=True)

# Formatear la columna del empleado con ceros a la izquierda (8 dígitos)
resumen_empleados['A quien se le paga'] = resumen_empleados['A quien se le paga'].astype(str).str.zfill(8)

# Dar formato de moneda
formato_moneda = lambda x: f"$ {x:,.2f}" if pd.notnull(x) and x != 0 else "$ -   "
columnas_dinero = ['Pago Base E', 'PAGO BOOSTER E', 'TOTAL SEMANAL']

for col in columnas_dinero:
    resumen_empleados[col] = resumen_empleados[col].apply(formato_moneda)

# Visualizar
print(resumen_empleados.to_string(index=False))

A quien se le paga Pago Base E PAGO BOOSTER E TOTAL SEMANAL
          00370115  $ 2,429.66       $ 550.61    $ 2,980.27
          00370169  $ 3,230.63       $ 688.06    $ 3,918.69
          00371936  $ 1,505.26        $ 96.92    $ 1,602.18
          10510050    $ 770.80        $ 77.53      $ 848.33
          10514641    $ 163.76         $ -         $ 163.76
          10544175    $ 531.39        $ 77.53      $ 608.92
